# Piyu AI Fashion Design Generator — FINAL

**Repository:** https://github.com/Piyu242005/AI-Fashion-Design-Generator-IBM-INTERSHIP-2026  
**Models:** https://huggingface.co/Piyu2420/AI-Fashion-Design-Generator-IBM-INTERSHIP-2026

## Pipeline

```
USER TEXT PROMPT
      ↓
STAGE 1 — RealVisXL V4.0 Lightning (Diffusers / SDXL)
      ↓
Photorealistic human fashion model image
      ↓
STAGE 2 — IDM-VTON  (optional — set RUN_IDM_VTON = True)
      ↓
Virtual try-on / clothing transfer output
```

**Run all cells top-to-bottom on a fresh T4/L4 GPU runtime.**  
Stage 1 is independently runnable. Stage 2 (IDM-VTON) is disabled by default.

---
## Section 0 — Project Configuration

In [ ]:
# ============================================================
# Section 0 — Project Configuration
#
# All important paths and flags are defined here.
# Edit PROMPT and RUN_IDM_VTON before running.
# ============================================================

import os
import sys
from pathlib import Path

# ── Repository / project paths ───────────────────────────────────────────
# NOTE: Repository name contains 'INTERSHIP' (one 'N') — do not change.
REPO_NAME    = "AI-Fashion-Design-Generator-IBM-INTERSHIP-2026"
PROJECT_DIR  = "Piyu-AI-Clothing-Fashion-Design-Generator"

REPO_ROOT    = Path("/content") / REPO_NAME          # Colab clone location
PROJECT      = REPO_ROOT / PROJECT_DIR                # project root
SRC_DIR      = PROJECT / "src"
WEIGHTS_DIR  = PROJECT / "weights"
OUTPUT_DIR   = PROJECT / "reference_images"
RESULTS_DIR  = PROJECT / "results"

# ── Key file paths ───────────────────────────────────────────────────────
GENERATE_MODEL = PROJECT / "generate_model.py"
MODEL_MANAGER  = SRC_DIR  / "model_manager.py"
MODEL_WEIGHTS  = WEIGHTS_DIR / "realvisxl.safetensors"
MODEL_IMAGE    = OUTPUT_DIR  / "fashion_model.png"

# ── User-editable generation settings ───────────────────────────────────
# Replace PROMPT with your clothing description.
# Keep it concise — SDXL's text encoder limit is ~75 tokens.
PROMPT = "a modern black crop top and high waist skirt"

# Example male model prompt:
# PROMPT = "a young male model wearing an oversized black shirt and tailored black trousers"

STEPS          = 4       # 4 steps recommended for Lightning distilled models
WIDTH          = 768     # output width in pixels
HEIGHT         = 1024    # output height in pixels
GUIDANCE_SCALE = 0.0     # 0.0 = Lightning CFG-free distillation
SEED           = None    # None = random seed; set an int for reproducibility

# ── IDM-VTON Stage 2 flag ────────────────────────────────────────────────
# Set to True ONLY if IDM-VTON deps are installed and verified.
# When False, ALL IDM-VTON and DensePose imports are skipped.
RUN_IDM_VTON = False

# ── Print summary ────────────────────────────────────────────────────────
print("Project Configuration")
print("=" * 60)
print(f"  REPO_ROOT       : {REPO_ROOT}")
print(f"  PROJECT         : {PROJECT}")
print(f"  GENERATE_MODEL  : {GENERATE_MODEL}")
print(f"  MODEL_MANAGER   : {MODEL_MANAGER}")
print(f"  MODEL_WEIGHTS   : {MODEL_WEIGHTS}")
print(f"  OUTPUT_DIR      : {OUTPUT_DIR}")
print(f"  MODEL_IMAGE     : {MODEL_IMAGE}")
print()
print(f"  PROMPT          : {PROMPT}")
print(f"  STEPS           : {STEPS}")
print(f"  WIDTH x HEIGHT  : {WIDTH} x {HEIGHT}")
print(f"  GUIDANCE_SCALE  : {GUIDANCE_SCALE}")
print(f"  SEED            : {SEED}")
print(f"  RUN_IDM_VTON    : {RUN_IDM_VTON}")
print("=" * 60)

---
## Section 1 — Environment Information

In [ ]:
# ============================================================
# Section 1 — Environment Information
# ============================================================

import sys
import subprocess

print("Environment Information")
print("=" * 60)
print(f"Python version  : {sys.version}")

try:
    import torch
    print(f"PyTorch version : {torch.__version__}")
    print(f"CUDA available  : {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU name        : {torch.cuda.get_device_name(0)}")
        print(f"CUDA version    : {torch.version.cuda}")
        total_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        print(f"GPU memory      : {total_mem:.1f} GB")
    else:
        print("GPU name        : N/A (CPU only — inference will be slow)")
        print("CUDA version    : N/A")
except ImportError:
    print("PyTorch version : NOT INSTALLED — run Section 2 first")

try:
    import torchvision
    print(f"Torchvision     : {torchvision.__version__}")
except ImportError:
    print("Torchvision     : NOT INSTALLED")

print()
result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
else:
    print("nvidia-smi: not available (no GPU or driver not loaded)")
print("=" * 60)

---
## Section 2 — Controlled Dependency Installation

**Rules applied here:**
- PyTorch / Torchvision are **not** reinstalled — Colab's pre-installed CUDA build is used.
- Each package is installed exactly **once** with a pinned or minimum version.
- IDM-VTON / DensePose are **not** installed here unless `RUN_IDM_VTON = True`.
- `auto1111sdk`, `clip`, `torchdiffeq`, `torchsde`, `cleanfid` are **not** installed.
- `numpy==1.26.4` is pinned once and not reinstalled in any later cell.

In [ ]:
# ============================================================
# Section 2 — Controlled Dependency Installation
#
# Rules:
#  • PyTorch / Torchvision are NEVER reinstalled.
#  • numpy is upgraded to >=2.1 FIRST — see why below.
#  • scipy is reinstalled AFTER numpy so its wheel matches.
#  • Pillow is NOT capped at <11.
#  • IDM-VTON / DensePose NOT installed unless RUN_IDM_VTON = True.
#  • auto1111sdk, clip, torchdiffeq, torchsde, cleanfid NOT installed.
#
# WHY numpy must be >=2.1
# -----------------------
# Colab currently ships numpy 2.0.2.  The pre-installed scipy wheel was
# compiled against numpy 2.1, which added
# numpy._core._multiarray_umath._blas_supports_fpe.
# When numpy is 2.0.x that symbol is absent, causing:
#   scipy.linalg._cythonized_array_utils  → AttributeError
#   scipy.optimize                        → fails to import
#   transformers/loss_for_object_detection → import chain breaks
#   CLIPImageProcessor                    → ModuleNotFoundError
#   StableDiffusionXLPipeline             → RuntimeError
# Fix: upgrade numpy to 2.1+ so the existing scipy binary works.
# Then reinstall scipy to guarantee a fresh, matching wheel.
# A runtime restart is required afterward.
# ============================================================

import subprocess, sys

def pip_install(packages: list, quiet: bool = False):
    """Run pip install for a list of package specs."""
    cmd = [sys.executable, "-m", "pip", "install"] + packages
    if quiet:
        cmd.append("-q")
    result = subprocess.run(cmd, capture_output=False)
    if result.returncode != 0:
        raise RuntimeError(f"pip install failed for: {packages}")

import numpy as _np_check
_np_ver = tuple(int(x) for x in _np_check.__version__.split('.')[:2])

print("Installing Stage 1 dependencies...")
print(f"  Current numpy : {_np_check.__version__}")
print()

# ── 0. Fix numpy/scipy ABI mismatch FIRST ────────────────────────────────
# Must happen before anything else because scipy is imported transitively
# by transformers the moment CLIPImageProcessor is touched.
if _np_ver < (2, 1):
    print("[0/5] numpy <2.1 detected — upgrading to >=2.1 to match scipy ABI")
    pip_install(["numpy>=2.1"], quiet=True)
    print("[0/5] Reinstalling scipy against upgraded numpy")
    pip_install(["scipy", "--force-reinstall"], quiet=True)
    print()
    print("=" * 60)
    print("RUNTIME RESTART REQUIRED")
    print("numpy was upgraded.  You MUST restart the Colab runtime now:")
    print("  Runtime > Restart runtime  (Ctrl+M .)")
    print("Then run all cells again from the top.")
    print("=" * 60)
    # Trigger Colab's built-in restart dialog if available
    try:
        import google.colab.runtime
        google.colab.runtime.unassign()
    except Exception:
        pass
    raise SystemExit(
        "Runtime restart required after numpy upgrade. "
        "Restart and re-run from the top."
    )
else:
    print(f"[0/5] numpy {_np_check.__version__} >= 2.1 — scipy ABI check OK, skipping upgrade")

# ── 1. Diffusers ecosystem ────────────────────────────────────────────────
print("[1/5] diffusers / transformers / accelerate / safetensors / tokenizers / peft")
pip_install([
    "diffusers>=0.29.0",
    "transformers>=4.40.0",
    "accelerate>=0.30.0",
    "safetensors>=0.4.3",
    "tokenizers>=0.19.0",
    "peft>=0.10.0",
], quiet=True)

# ── 2. Hugging Face Hub ───────────────────────────────────────────────────
print("[2/5] huggingface_hub")
pip_install(["huggingface_hub>=0.23.0"], quiet=True)

# ── 3. Image processing ───────────────────────────────────────────────────
print("[3/5] Pillow / opencv-python")
pip_install(["Pillow>=9.5.0", "opencv-python>=4.9.0"], quiet=True)

# ── 4. Utilities ──────────────────────────────────────────────────────────
print("[4/5] tqdm / python-dotenv")
pip_install(["tqdm", "python-dotenv>=1.0.0"], quiet=True)

print()
print("Stage 1 dependencies installed.")
print()

# ── 5. IDM-VTON dependencies (skipped by default) ────────────────────────
if RUN_IDM_VTON:
    print("[5/5] RUN_IDM_VTON = True — installing Stage 2 dependencies...")
    pip_install([
        "open-clip-torch>=2.20.0",
        "onnxruntime>=1.16.3",
        "einops>=0.7.0",
        "kornia>=0.6.7",
        "timm>=0.9.16",
        "omegaconf>=2.2.3",
    ], quiet=True)
    print("  DensePose / Detectron2 must be built from source — see requirements-idm-vton.txt")
else:
    print("[5/5] RUN_IDM_VTON = False — IDM-VTON / DensePose skipped.")

print()
print("Section 2 complete.")

---
## Section 3 — Dependency Verification

In [ ]:
# ============================================================
# Section 3 — Dependency Verification
#
# Verifies all Stage 1 imports and prints a pass/fail table.
# Catches both ImportError AND RuntimeError — diffusers/transformers
# lazy-import wrappers raise RuntimeError, not ImportError.
#
# Also runs a direct scipy smoke test.  If scipy fails here it means
# numpy was not yet upgraded to >=2.1.  Go back to Section 2.
# ============================================================

import importlib
import numpy as np

# ── scipy ABI pre-check ───────────────────────────────────────────────────
# This must succeed before we try to import StableDiffusionXLPipeline.
# If scipy fails here, the root cause is numpy <2.1 vs scipy >=1.15 ABI.
print("Pre-check: numpy/scipy ABI")
print(f"  numpy version : {np.__version__}")
_np_ver = tuple(int(x) for x in np.__version__.split('.')[:2])
if _np_ver < (2, 1):
    raise RuntimeError(
        f"numpy {np.__version__} is below 2.1.\n"
        "Section 2 should have upgraded numpy and prompted a runtime restart.\n"
        "Go back to Section 2 and follow the restart instructions."
    )

try:
    from scipy.optimize import linear_sum_assignment as _lsa  # noqa: F401
    import scipy as _scipy
    print(f"  scipy version : {_scipy.__version__} — ABI OK")
except Exception as _e:
    raise RuntimeError(
        f"scipy failed to import: {_e}\n"
        "This is a numpy/scipy ABI mismatch.\n"
        "Fix: run Section 2 again (it will upgrade numpy and reinstall scipy),"
        " then restart the runtime."
    ) from _e

print()

# ── Full dependency table ─────────────────────────────────────────────────
checks = [
    ("torch",           "torch"),
    ("torchvision",     "torchvision"),
    ("diffusers",       "diffusers"),
    ("transformers",    "transformers"),
    ("accelerate",      "accelerate"),
    ("safetensors",     "safetensors"),
    ("PIL",             "Pillow"),
    ("numpy",           "numpy"),
    ("huggingface_hub", "huggingface_hub"),
]

print("Dependency Verification")
print(f"{'Package':<20} {'Status':<10} {'Version'}")
print("-" * 55)

failures = []
for import_name, display_name in checks:
    try:
        mod = importlib.import_module(import_name)
        ver = getattr(mod, "__version__", "(no version attr)")
        print(f"  {display_name:<18} {'PASS':<8} {ver}")
    except (ImportError, RuntimeError) as e:
        print(f"  {display_name:<18} {'FAIL':<8} {e}")
        failures.append(display_name)

# ── StableDiffusionXLPipeline deep import check ───────────────────────────
# Triggers the full diffusers → transformers → CLIPImageProcessor chain.
# After the scipy fix above this should always pass.
try:
    from diffusers import StableDiffusionXLPipeline
    print(f"  {'StableDiffXLPipe':<18} {'PASS':<8} available")
except (ImportError, RuntimeError) as e:
    short = str(e)[:150]
    print(f"  {'StableDiffXLPipe':<18} {'FAIL':<8} {short}")
    failures.append("StableDiffusionXLPipeline")

print()
if failures:
    raise RuntimeError(
        f"Stage 1 dependency failures: {failures}\n"
        "If StableDiffusionXLPipeline failed:\n"
        "  1. The scipy/numpy ABI check above passed, so the issue is\n"
        "     likely a stale in-process module.\n"
        "  2. Restart the runtime: Runtime > Restart runtime\n"
        "  3. Run all cells again from the top."
    )
print("All Stage 1 dependencies verified.")

---
## Section 4 — Project Structure Verification

In [ ]:
# ============================================================
# Section 4 — Project Structure Verification
#
# Clone the repository if not already present, then verify
# all critical files and directories.
# ============================================================

import subprocess
from pathlib import Path

# ── Clone repository if needed ───────────────────────────────────────────
GITHUB_URL = "https://github.com/Piyu242005/AI-Fashion-Design-Generator-IBM-INTERSHIP-2026.git"

if not REPO_ROOT.exists():
    print(f"Cloning repository to {REPO_ROOT} ...")
    result = subprocess.run(
        ["git", "clone", GITHUB_URL, str(REPO_ROOT)],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        raise RuntimeError(f"git clone failed:\n{result.stderr}")
    print("✅ Repository cloned.")
else:
    print(f"✅ Repository already present at {REPO_ROOT}")

# ── Ensure output directories exist ─────────────────────────────────────
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

# ── Add project root and src/ to sys.path ────────────────────────────────
for p in [str(PROJECT), str(SRC_DIR)]:
    if p not in sys.path:
        sys.path.insert(0, p)

# ── Verify structure ─────────────────────────────────────────────────────
items_to_check = [
    ("generate_model.py",     GENERATE_MODEL,            True),
    ("src/",                  SRC_DIR,                   True),
    ("src/model_manager.py",  MODEL_MANAGER,             True),
    ("weights/",              WEIGHTS_DIR,               True),
    ("reference_images/",     OUTPUT_DIR,                True),
    ("idm_vton/",             PROJECT / "idm_vton",      False),  # optional
    ("src/__init__.py",       SRC_DIR / "__init__.py",   True),
]

print()
print("Project Structure")
print(f"{'Path':<30} {'Status':<10} {'Required'}")
print("-" * 55)

critical_missing = []
for label, path, required in items_to_check:
    exists = path.exists()
    status = "✅ found" if exists else ("❌ MISSING" if required else "⚠️  absent")
    req_str = "required" if required else "optional"
    print(f"  {label:<28} {status:<12} {req_str}")
    if required and not exists:
        critical_missing.append(str(path))

print()
if critical_missing:
    raise RuntimeError(
        f"Critical project files missing:\n" +
        "\n".join(f"  {p}" for p in critical_missing) +
        "\n\nCheck that the repository cloned correctly."
    )
print("✅ Project structure OK.")

---
## Section 5 — RealVisXL Model Verification

In [ ]:
# ============================================================
# Section 5 — RealVisXL Model Verification
#
# Uses src/model_manager.get_realvisxl_path() to resolve the
# local path for the RealVisXL V4.0 Lightning safetensors file.
#
# If already cached at weights/realvisxl.safetensors, the
# download is skipped. No duplicate download logic here.
# ============================================================

import os
from pathlib import Path

# Optional: set HF_TOKEN if the repository is private
# os.environ["HF_TOKEN"] = "hf_YOUR_TOKEN_HERE"

# Set MODEL_DIR so model_manager resolves paths relative to PROJECT
os.environ["MODEL_DIR"] = str(PROJECT)

print("Resolving RealVisXL model weight...")
print(f"  Expected location : {MODEL_WEIGHTS}")

if MODEL_WEIGHTS.exists() and MODEL_WEIGHTS.stat().st_size > 0:
    size_gb = MODEL_WEIGHTS.stat().st_size / (1024**3)
    print(f"  Status            : ✅ Cached")
    print(f"  File size         : {size_gb:.2f} GB")
    realvisxl_path = str(MODEL_WEIGHTS)
else:
    print(f"  Status            : ⬇️  Not cached — downloading from Hugging Face")
    print(f"  This may take several minutes on first run.")

    try:
        from src.model_manager import get_realvisxl_path
        realvisxl_path = get_realvisxl_path()
        size_gb = Path(realvisxl_path).stat().st_size / (1024**3)
        print(f"  ✅ Downloaded: {realvisxl_path} ({size_gb:.2f} GB)")
    except Exception as e:
        raise RuntimeError(
            f"Failed to download RealVisXL: {e}\n"
            "Check your HF_TOKEN if the repo is private, "
            "or verify the HF repo ID: Piyu2420/AI-Fashion-Design-Generator-IBM-INTERSHIP-2026"
        )

print()
print(f"  realvisxl_path    : {realvisxl_path}")
print("✅ RealVisXL model ready.")

---
## Section 6 — Load generate_model.py (Diffusers Implementation)

In [ ]:
# ============================================================
# Section 6 — Load generate_model.py
#
# Verifies the CURRENT Diffusers implementation:
#   ✅ StableDiffusionXLPipeline present
#   ✅ auto1111sdk NOT used
# Stops with a clear error if the old Auto1111SDK version is detected.
# ============================================================

import importlib
import importlib.util
from pathlib import Path

print(f"Verifying generate_model.py at: {GENERATE_MODEL}")
print()

# ── 1. Verify file exists ─────────────────────────────────────────────────
if not GENERATE_MODEL.exists():
    raise FileNotFoundError(
        f"generate_model.py not found at {GENERATE_MODEL}\n"
        "Make sure Section 4 completed successfully."
    )
print(f"  ✅ File exists : {GENERATE_MODEL}")

# ── 2. Read source and verify implementation ──────────────────────────────
source = GENERATE_MODEL.read_text(encoding="utf-8")

# Guard: reject old Auto1111SDK implementation
if "auto1111sdk" in source:
    raise RuntimeError(
        "FATAL: generate_model.py contains 'auto1111sdk'.\n"
        "This is the OLD implementation. The current version must use Diffusers.\n"
        "Pull the latest version from the repository and retry."
    )
print("  ✅ auto1111sdk : NOT present (correct)")

# Verify new Diffusers implementation
if "StableDiffusionXLPipeline" not in source:
    raise RuntimeError(
        "FATAL: generate_model.py does NOT contain 'StableDiffusionXLPipeline'.\n"
        "Expected the Diffusers SDXL implementation. Pull the latest version."
    )
print("  ✅ StableDiffusionXLPipeline : present (correct)")

# Verify generate() function signature
if "def generate(" not in source:
    raise RuntimeError(
        "FATAL: generate_model.py does not define a generate() function."
    )
print("  ✅ generate()  : function present (correct)")

# ── 3. Import the module ──────────────────────────────────────────────────
print()
print("Importing generate_model module...")

spec = importlib.util.spec_from_file_location("generate_model", str(GENERATE_MODEL))
generate_model = importlib.util.module_from_spec(spec)

# Ensure the module can find src/model_manager
import sys
for p in [str(PROJECT), str(SRC_DIR)]:
    if p not in sys.path:
        sys.path.insert(0, p)

try:
    spec.loader.exec_module(generate_model)
except Exception as e:
    raise RuntimeError(
        f"Failed to import generate_model.py: {e}\n"
        "Check that Stage 1 dependencies (Section 2) are installed."
    )

print("  ✅ generate_model imported successfully.")
print()

# ── 4. Scheduler compatibility note ──────────────────────────────────────
# generate_model.py uses DPMSolverSinglestepScheduler with a try/except:
# if it fails on this diffusers version, it keeps the default scheduler.
# This was added to prevent 'step must be greater than zero' with old builds.
# diffusers>=0.29.0 (installed in Section 2) resolves this issue.
try:
    from diffusers import DPMSolverSinglestepScheduler, EulerDiscreteScheduler
    import diffusers
    print(f"  Diffusers version          : {diffusers.__version__}")
    print(f"  DPMSolverSinglestepScheduler: available")
    print(f"  EulerDiscreteScheduler     : available (fallback)")
except ImportError as e:
    print(f"  ⚠️  Scheduler import issue: {e}")

print()
print("✅ Section 6 complete — generate_model ready.")

---
## Section 7 — Generate Fashion Model (Stage 1)

Calls `generate_model.generate()` directly (no subprocess).  
Output is saved to `reference_images/fashion_model.png` and displayed inline.

In [ ]:
# ============================================================
# Section 7 — Generate Fashion Model (Stage 1)
#
# Calls generate_model.generate() — NOT subprocess.
# Parameters: PROMPT, STEPS, WIDTH, HEIGHT, GUIDANCE_SCALE, SEED
# Output: reference_images/fashion_model.png
# ============================================================

import gc
import time
from pathlib import Path
from IPython.display import display
from PIL import Image

# ── Ensure output directory exists ───────────────────────────────────────
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Stage 1 — RealVisXL V4.0 Lightning")
print("=" * 60)
print(f"  Prompt         : {PROMPT}")
print(f"  Steps          : {STEPS}")
print(f"  Resolution     : {WIDTH} x {HEIGHT}")
print(f"  Guidance scale : {GUIDANCE_SCALE}")
print(f"  Seed           : {SEED if SEED is not None else 'random'}")
print(f"  Output path    : {MODEL_IMAGE}")
print("=" * 60)
print()

start_time = time.time()

try:
    generate_model.generate(
        clothing_type  = PROMPT,
        output_path    = str(MODEL_IMAGE),
        steps          = STEPS,
        width          = WIDTH,
        height         = HEIGHT,
        guidance_scale = GUIDANCE_SCALE,
        seed           = SEED,
    )
except Exception as e:
    print(f"\n❌ Stage 1 failed: {e}")
    raise

elapsed = time.time() - start_time
print()
print(f"✅ Stage 1 complete in {elapsed:.1f}s")

# ── Display generated image ───────────────────────────────────────────────
if MODEL_IMAGE.exists():
    size_kb = MODEL_IMAGE.stat().st_size / 1024
    print(f"   Saved to : {MODEL_IMAGE} ({size_kb:.1f} KB)")
    print()
    print("Stage 1 Output:")
    display(Image.open(MODEL_IMAGE))
else:
    raise FileNotFoundError(
        f"Output image not found at {MODEL_IMAGE}\n"
        "Generation may have silently failed."
    )

---
## Section 8 — IDM-VTON Stage 2 (Optional)

Set `RUN_IDM_VTON = True` in Section 0 to enable.  
When `False`, all IDM-VTON and DensePose code is completely skipped.

In [ ]:
# ============================================================
# Section 8a — IDM-VTON Compatibility Check
#
# ONLY runs when RUN_IDM_VTON = True.
# When False: prints a skip message and exits immediately.
# ============================================================

if not RUN_IDM_VTON:
    print("IDM-VTON skipped — Stage 1 only.")
    print()
    print("To enable Stage 2:")
    print("  1. Set RUN_IDM_VTON = True in Section 0.")
    print("  2. Install requirements-idm-vton.txt dependencies.")
    print("  3. Build Detectron2 and DensePose from source for Python 3.12.")
    print("  4. Restart the runtime and run from the top.")
else:
    print("RUN_IDM_VTON = True — performing compatibility checks...")
    print()

    issues = []

    # ── 1. NumPy check ────────────────────────────────────────────────────
    import numpy as np_check
    np_ver = np_check.__version__
    if not np_ver.startswith("1.26"):
        issues.append(
            f"NumPy {np_ver} — IDM-VTON requires numpy==1.26.x to avoid "
            "'numpy.dtype size changed' binary incompatibility with Detectron2."
        )
        print(f"  ❌ NumPy  : {np_ver} (want 1.26.x)")
    else:
        print(f"  ✅ NumPy  : {np_ver}")

    # ── 2. Detectron2 check ───────────────────────────────────────────────
    try:
        import detectron2
        print(f"  ✅ Detectron2 : {detectron2.__version__}")
    except ImportError:
        issues.append(
            "detectron2 not installed. Build from source:\n"
            "  pip install git+https://github.com/facebookresearch/detectron2.git"
        )
        print("  ❌ Detectron2 : not installed")

    # ── 3. DensePose check ────────────────────────────────────────────────
    try:
        from densepose import add_densepose_config
        print("  ✅ DensePose  : available")
    except ImportError:
        issues.append(
            "densepose not installed. Build from source:\n"
            "  pip install 'git+https://github.com/facebookresearch/detectron2.git"
            "#subdirectory=projects/DensePose'"
        )
        print("  ❌ DensePose  : not installed")

    # ── 4. idm_vton/ directory check ──────────────────────────────────────
    idm_dir = PROJECT / "idm_vton"
    if not idm_dir.exists():
        issues.append(
            f"idm_vton/ directory not found at {idm_dir}. "
            "Ensure the full repository is cloned."
        )
        print(f"  ❌ idm_vton/  : not found")
    else:
        print(f"  ✅ idm_vton/  : {idm_dir}")

    print()
    if issues:
        print("⚠️  IDM-VTON compatibility issues detected:")
        for i, issue in enumerate(issues, 1):
            print(f"  [{i}] {issue}")
        print()
        print("Stage 1 output is still valid. Fix the issues above before running Stage 2.")
        # Set flag to prevent Stage 2 execution
        _idmvton_ready = False
    else:
        print("✅ IDM-VTON compatibility check passed.")
        _idmvton_ready = True

In [ ]:
# ============================================================
# Section 8b — IDM-VTON Execution
#
# ONLY executes when RUN_IDM_VTON = True AND all checks pass.
# ============================================================

TRYON_IMAGE = RESULTS_DIR / "final_tryon.png"

if not RUN_IDM_VTON:
    print("IDM-VTON skipped — Stage 1 only.")
elif not _idmvton_ready:
    print("IDM-VTON skipped — compatibility issues found in Section 8a.")
    print("Resolve the issues above before running Stage 2.")
else:
    print("Stage 2 — IDM-VTON Virtual Try-On")
    print("=" * 60)

    # Add idm_vton/src to sys.path
    idm_src = PROJECT / "idm_vton" / "src"
    for p in [str(PROJECT / "idm_vton"), str(idm_src)]:
        if p not in sys.path:
            sys.path.insert(0, p)

    try:
        import importlib
        tryon = importlib.import_module("try_on")

        # Default garment image (replace with actual garment path)
        GARMENT_IMAGE = PROJECT / "samples" / "garment.png"

        if not GARMENT_IMAGE.exists():
            raise FileNotFoundError(
                f"Garment image not found: {GARMENT_IMAGE}\n"
                "Place a garment PNG at samples/garment.png, then re-run."
            )

        tryon.run_tryon(
            model_image_path   = str(MODEL_IMAGE),
            garment_image_path = str(GARMENT_IMAGE),
            output_path        = str(TRYON_IMAGE),
        )

        if TRYON_IMAGE.exists():
            print(f"✅ Stage 2 complete: {TRYON_IMAGE}")
            from IPython.display import display
            from PIL import Image
            display(Image.open(TRYON_IMAGE))
        else:
            raise FileNotFoundError(f"IDM-VTON output not found: {TRYON_IMAGE}")

    except Exception as e:
        print(f"\n❌ Stage 2 (IDM-VTON) failed: {e}")
        print("   Stage 1 output is unaffected.")
        raise

---
## Section 9 — Final Output Summary

In [ ]:
# ============================================================
# Section 9 — Final Output Summary
#
# Display Stage 1 image (always).
# Display Stage 2 image if available.
# Print all output paths.
# ============================================================

from IPython.display import display
from PIL import Image

print("Final Output Summary")
print("=" * 60)

# ── Stage 1 ───────────────────────────────────────────────────────────────
if MODEL_IMAGE.exists():
    size_kb = MODEL_IMAGE.stat().st_size / 1024
    print(f"  Stage 1 (Fashion Model) : {MODEL_IMAGE}")
    print(f"  File size               : {size_kb:.1f} KB")
    print()
    print("Stage 1 — RealVisXL Output:")
    display(Image.open(MODEL_IMAGE))
else:
    print(f"  Stage 1 : ❌ not found at {MODEL_IMAGE}")

# ── Stage 2 ───────────────────────────────────────────────────────────────
print()
TRYON_IMAGE = RESULTS_DIR / "final_tryon.png"
if TRYON_IMAGE.exists():
    size_kb = TRYON_IMAGE.stat().st_size / 1024
    print(f"  Stage 2 (Try-On Result) : {TRYON_IMAGE}")
    print(f"  File size               : {size_kb:.1f} KB")
    print()
    print("Stage 2 — IDM-VTON Output:")
    display(Image.open(TRYON_IMAGE))
else:
    if RUN_IDM_VTON:
        print(f"  Stage 2 : ⚠️  not found (IDM-VTON may have failed)")
    else:
        print(f"  Stage 2 : — (RUN_IDM_VTON = False)")

print()
print("=" * 60)
print(f"  Output directory : {OUTPUT_DIR}")
print(f"  Results directory: {RESULTS_DIR}")
print("=" * 60)

---
## Section 10 — GPU Memory Cleanup

In [ ]:
# ============================================================
# Section 10 — GPU Memory Cleanup
#
# Release GPU memory after all stages are complete.
# Safe to run at any point.
# ============================================================

import gc
import torch

print("GPU Memory Cleanup")
print("=" * 60)

if torch.cuda.is_available():
    mem_before = torch.cuda.memory_allocated(0) / (1024**2)
    print(f"  GPU memory allocated before cleanup : {mem_before:.1f} MB")

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    mem_after = torch.cuda.memory_allocated(0) / (1024**2)
    print(f"  GPU memory allocated after cleanup  : {mem_after:.1f} MB")
    print(f"  ✅ GPU cache cleared.")
else:
    print("  GPU not available — nothing to clear.")

print("=" * 60)
print()
print("✅ Pipeline complete.")
print()
print("Summary:")
print(f"  PROMPT       : {PROMPT}")
print(f"  Stage 1 out  : {MODEL_IMAGE}")
print(f"  Stage 2 out  : {'enabled' if RUN_IDM_VTON else 'skipped (RUN_IDM_VTON=False)'}")
print(f"  Repo         : https://github.com/Piyu242005/AI-Fashion-Design-Generator-IBM-INTERSHIP-2026")